# Merge 2022–2025 Data into Main CSVs

Appends all new 2022-2025 files into the existing final CSVs.

**Files merged:**
- `paper_nodes.csv` ← `paper_nodes_2022_2025.csv`
- `openalex_metadata_full.csv` ← `openalex_metadata_2022_2025.csv`
- `entity_nodes.csv` ← `entity_nodes_2022_2025.csv`
- `knowledge_edges.csv` ← `knowledge_edges_2022_2025.csv`

**Note:** `citation_edges_2022_2025.csv` is used separately for citation signal in novelty scoring.

In [15]:
import pandas as pd

OUT_DIR = "../outputs/final/"
INT_DIR = "../outputs/intermediate/"

print("Setup complete.")

Setup complete.


In [19]:
# 1. paper_nodes
old_pn = pd.read_csv(OUT_DIR + 'paper_nodes.csv')
new_pn = pd.read_csv(OUT_DIR + 'paper_nodes_2022_2025.csv')

print(f"Old : {len(old_pn):,}")
print(f"New : {len(new_pn):,}")

merged_pn = pd.concat([old_pn, new_pn], ignore_index=True)
merged_pn = merged_pn.drop_duplicates(subset='node_id').reset_index(drop=True)

assert merged_pn['node_id'].is_unique
print(f"Merged : {len(merged_pn):,}")
print(merged_pn['split'].value_counts().to_string())
print("Year range:", merged_pn['year'].min(), '–', merged_pn['year'].max())

merged_pn.to_csv(OUT_DIR + 'paper_nodes.csv', index=False)
print("Paper_nodes.csv updated")

Old : 4,860
New : 2,331
Merged : 4,860
split
SKG      3166
NOVEL     854
BLOG      840
Year range: 2010.0 – 2025.0
Paper_nodes.csv updated


In [7]:
# 2. openalex_metadata
old_meta = pd.read_csv(INT_DIR+'openalex_metadata_full.csv')
new_meta = pd.read_csv(INT_DIR + 'openalex_metadata_2022_2025.csv')

print(f"Old : {len(old_meta):,}")
print(f"New : {len(new_meta):,}")

merged_meta = pd.concat([old_meta, new_meta], ignore_index=True)
merged_meta = merged_meta.drop_duplicates(subset='global_paper_id').reset_index(drop=True)

print(f"Merged : {len(merged_meta):,}")
print(f"With abstracts    : {merged_meta['abstract'].notna().sum():,}")
print(f"Without abstracts : {merged_meta['abstract'].isna().sum():,}")

merged_meta.to_csv(INT_DIR + 'openalex_metadata_full.csv', index=False)
print("Openalex_metadata_full.csv updated")

Old : 2,529
New : 2,331
Merged : 4,860
With abstracts    : 4,242
Without abstracts : 618
Openalex_metadata_full.csv updated


In [9]:
# 3. entity_nodes
old_en = pd.read_csv(OUT_DIR + 'entity_nodes.csv')
new_en = pd.read_csv(OUT_DIR + 'entity_nodes_2022_2025.csv')

print(f"Old : {len(old_en):,}")
print(f"New : {len(new_en):,}")

merged_en = pd.concat([old_en, new_en], ignore_index=True)
merged_en = merged_en.drop_duplicates(subset='node_id').reset_index(drop=True)

print(f"Merged : {len(merged_en):,}")

merged_en.to_csv(OUT_DIR + 'entity_nodes.csv', index=False)
print("entity_nodes.csv updated")

Old : 293,149
New : 184,333
Merged : 293,149
entity_nodes.csv updated


In [10]:
# 4. knowledge_edges
old_ke = pd.read_csv(OUT_DIR + 'knowledge_edges.csv')
new_ke = pd.read_csv(OUT_DIR + 'knowledge_edges_2022_2025.csv')

print(f"Old : {len(old_ke):,}")
print(f"New : {len(new_ke):,}")

merged_ke = pd.concat([old_ke, new_ke], ignore_index=True)
merged_ke = merged_ke.drop_duplicates().reset_index(drop=True)

print(f"Merged : {len(merged_ke):,}")
print(f"Year range : {merged_ke['year'].min()} – {merged_ke['year'].max()}")

orphans = ~merged_ke['source'].isin(merged_pn['node_id'])
print(f"Edges with unknown source : {orphans.sum():,}")

merged_ke.to_csv(OUT_DIR + 'knowledge_edges.csv', index=False)
print("Knowledge_edges.csv updated")

Old : 286,708
New : 290,316
Merged : 577,024
Year range : 2010.0 – 2025.0
Edges with unknown source : 0
Knowledge_edges.csv updated


In [13]:
# 5. Final summary
print("MERGED DATASET SUMMARY")
print(f"paper_nodes.csv            : {len(merged_pn):>8,} papers")
print(f"openalex_metadata_full.csv : {len(merged_meta):>8,} papers")
print(f"entity_nodes.csv           : {len(merged_en):>8,} entities")
print(f"knowledge_edges.csv        : {len(merged_ke):>8,} edges")

print("Split breakdown:")
print(merged_pn['split'].value_counts().to_string())
print("Year range (papers):", merged_pn['year'].min(), '–', merged_pn['year'].max())
print("Year range (edges) :", merged_ke['year'].min(), '–', merged_ke['year'].max())
no_edges = len(set(merged_pn['node_id']) - set(merged_ke['source']))
print(f"Papers with no knowledge edges : {no_edges:,}")

MERGED DATASET SUMMARY
paper_nodes.csv            :    4,860 papers
openalex_metadata_full.csv :    4,860 papers
entity_nodes.csv           :  293,149 entities
knowledge_edges.csv        :  577,024 edges
Split breakdown:
split
SKG      3166
NOVEL     854
BLOG      840
Year range (papers): 2010.0 – 2025.0
Year range (edges) : 2010.0 – 2025.0
Papers with no knowledge edges : 1,458


In [14]:
import pandas as pd
pn = pd.read_csv('../outputs/final/paper_nodes.csv')
print(pn['year'].value_counts().sort_index().tail(10))
print(pn['split'].value_counts())

year
2016.0     140
2017.0     221
2018.0     291
2019.0     437
2020.0     505
2021.0     404
2022.0    1040
2023.0    1091
2024.0     184
2025.0      21
Name: count, dtype: int64
split
SKG      3166
NOVEL     854
BLOG      840
Name: count, dtype: int64
